# 강의 03 · 실습 1 — 에이전트 동작 원리 · (5) 고난도 II

## 1. 문제상황

- 구름월드 고객센터에는 「지금 몇 시인지랑 환불 규정을 같이 알려 주세요」처럼 한 문장에 두 가지를 묻는 질문이 자주 들어옵니다.
- 지금의 안내 프로그램은 도구 호출 요청을 하나씩 차례로 실행하므로, 도구 하나가 느리면 뒤의 도구가 그만큼 기다립니다.
- 실제 FAQ 조회는 사내 시스템을 거쳐 약 2초가 걸리고, 시각 조회도 외부 시간 서버를 거쳐 약 2초가 걸립니다.
- 두 가지를 묻는 질문 하나에 두 도구의 지연이 그대로 더해져, 상담 화면에서 답이 늦게 나옵니다.
- 담당자는 한 응답에 실린 도구 호출 요청이 여러 개이면 한꺼번에 실행해서 답이 빨리 나오기를 원합니다.

## 2. 문제와 목표

- **문제**: 한 응답에 실린 도구 호출 요청 여러 개를 차례로 실행해, 도구마다 걸리는 시간이 더해집니다.
- **목표**
  - 한 응답에 실린 도구 호출 요청들을 한꺼번에 실행하고, 요청 순서대로 결과를 대화 기록에 붙이는 루프를 만듭니다.
  - 차례로 실행하는 판과 한꺼번에 실행하는 판을 같은 질문으로 돌려 소요 시간을 비교합니다.
    - 두 판: 차례 실행(요청을 하나씩 실행), 동시 실행(요청을 전부 제출한 뒤 순서대로 수거)
  - 주어지는 것: FAQ 사전 세 항목(운영시간·주차·환불)과 현재 시각 도구. 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
  - 두 도구 안에 2초 지연(`time.sleep(2)`)을 넣어 느린 도구를 흉내 냅니다.
  - 질문은 「지금 몇 시인지랑 환불 규정을 같이 알려 주세요.」 하나이고, 반복 상한은 4회입니다.
  - 한꺼번에 실행하는 판에서 스레드 풀에 작업을 제출하는 꼴은 `pool.submit(TOOLS[c["name"]].invoke, c["args"])`입니다.
- **목표 달성 여부의 판정 기준**:
  - 두 가지를 묻는 질문에서 첫 호출의 응답에 도구 호출 요청이 두 개 실리고,
  - 한꺼번에 실행하는 판의 도구 실행 시간이 차례로 실행하는 판의 절반에 가까우며(약 2초 대 약 4초),
  - 두 판의 최종 답에 시각과 환불 규정이 모두 들어 있는 것을 출력에서 확인합니다.
  - 출력에는 판마다 도구 실행에 걸린 시간과 최종 답이 찍힙니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex01_s5_diagram.svg)

## 4. 단계별 요구사항

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 주어진 것

라이브러리 불러오기, `.env` 읽기, 모델 준비와 주어진 자료는 아래 셀에 있습니다. 셀을 고치지 않고 그대로 실행합니다.

- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, `OPENAI_API_KEY=발급받은_키` 한 줄만 넣습니다.

In [ ]:
import os
import time
from concurrent.futures import ThreadPoolExecutor

from datetime import datetime
from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

# 주어진 자료
FAQ = {
    "운영시간": "매일 09:30~21:00에 운영합니다.",
    "주차": "주차장은 4,000대 규모이며 최초 30분은 무료입니다.",
    "환불": "이용일 전날까지 전액 환불, 당일은 50% 환불입니다.",
}


In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 첫 호출의 응답에 도구 호출 요청이 두 개(시각 조회와 환불 조회) 실려 있습니다.
2. 차례로 실행하는 판의 도구 실행 시간은 약 4초이고, 한꺼번에 실행하는 판의 도구 실행 시간은 약 2초입니다. 두 도구가 같은 시간에 돌았다는 뜻입니다.
3. 두 판의 최종 답에 현재 시각과 환불 규정이 모두 들어 있습니다. 실행 방식이 바뀌어도 답의 내용은 같습니다.

세 가지가 모두 확인되면 완성입니다. 모델이 요청을 하나씩 나누어 보내는 날도 있습니다. 그때는 셀을 다시 실행합니다.